# Part B — 진단: TEST R² 음수 원인 (우선순위 1)
`run_lr.ipynb` 1차 결과에서 **TEST R² = −1.28**. 평균 예측보다 나쁜 상태 → 원인 진단.
저장된 `features_partB.csv`를 불러와 동일 split·적합 재현.

**진단 3종 + α**
- A. train/test 분포 변화 (log_views, 카테고리 구성)
- B. 카테고리 레벨 미스매치 (test에만 존재 → 예측 불안정)
- C. 극단 잔차 (특정 바이럴 영상이 RMSE 지배?)
- α. 랜덤 split 비교 (시간 split 자체의 영향 분리)

In [1]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from sklearn.metrics import r2_score, mean_squared_error

features = pd.read_csv("features_partB.csv", parse_dates=["publish_time"])
hour_labels = ["00-05","06-11","12-17","18-23"]
features["hour_bin"] = pd.Categorical(features["hour_bin"], categories=hour_labels)

title_feats = ["title_len","word_count","caps_ratio","exclaim_cnt",
               "question_cnt","has_number","has_bracket","tag_cnt"]
formula = ("log_views ~ title_len + word_count + caps_ratio + exclaim_cnt + "
           "question_cnt + has_number + has_bracket + tag_cnt + "
           "C(hour_bin, Treatment(reference='18-23')) + "
           "C(publish_weekday) + C(category)")

def time_split(d, frac=0.8):
    d = d.dropna(subset=["publish_time"]).sort_values("publish_time").reset_index(drop=True)
    cut = int(len(d)*frac)
    return d.iloc[:cut].copy(), d.iloc[cut:].copy()

train, test = time_split(features)
print(f"train {len(train)} | test {len(test)}")

train 4999 | test 1250


## A. train / test 분포 변화

In [2]:
print("[log_views 요약]")
desc = pd.concat([train["log_views"].describe(), test["log_views"].describe()], axis=1)
desc.columns = ["train","test"]
print(desc.round(3))
print("\n평균 차이:", round(test["log_views"].mean()-train["log_views"].mean(),3))

[log_views 요약]
          train      test
count  4999.000  1250.000
mean     12.732    14.296
std       1.804     1.221
min       6.328    11.711
25%      11.678    13.443
50%      12.864    14.133
75%      13.938    14.989
max      18.822    19.233

평균 차이: 1.564


In [3]:
# 카테고리 구성 비율 비교 (상위 차이)
comp = pd.DataFrame({
    "train_%": train["category"].value_counts(normalize=True)*100,
    "test_%":  test["category"].value_counts(normalize=True)*100,
}).fillna(0)
comp["diff"] = (comp["test_%"]-comp["train_%"]).round(2)
comp.round(2).sort_values("diff", key=abs, ascending=False).head(10)

,train_%,test_%,diff
category,,,
News & Politics,8.92,4.40,-4.52
Entertainment,24.72,29.04,4.32
Gaming,1.24,3.12,1.88
Science & Technology,6.24,4.56,-1.68
Music,12.40,13.76,1.36
Pets & Animals,2.42,1.36,-1.06
Autos & Vehicles,1.22,0.32,-0.90
Sports,6.90,7.76,0.86
Film & Animation,4.74,5.52,0.78


## B. 카테고리 레벨 미스매치

In [4]:
train_cats = set(train["category"].unique())
test_cats  = set(test["category"].unique())
only_test  = test_cats - train_cats
only_train = train_cats - test_cats
print("test에만 존재(예측 불안정 위험):", only_test)
print("train에만 존재:", only_train)
if only_test:
    print("test 중 미학습 카테고리 행 수:",
          test["category"].isin(only_test).sum())

test에만 존재(예측 불안정 위험): set()
train에만 존재: set()


## C. 극단 잔차 (test)

In [5]:
ols = smf.ols(formula, data=train).fit()

# 예측 가능한 행만 (미학습 카테고리 제외)
test_eval = test[~test["category"].isin(only_test)].copy()
pred = ols.predict(test_eval)
test_eval["pred"] = pred.values
test_eval["resid"] = test_eval["log_views"] - test_eval["pred"]

r2  = r2_score(test_eval["log_views"], test_eval["pred"])
rmse= np.sqrt(mean_squared_error(test_eval["log_views"], test_eval["pred"]))
print(f"[미학습 카테고리 제외 후 TEST] n={len(test_eval)}  R2={r2:.4f}  RMSE={rmse:.4f}")
print("\n[잔차 절댓값 상위 8건]")
cols = ["category","log_views","pred","resid"]
print(test_eval.reindex(test_eval["resid"].abs().sort_values(ascending=False).index)[cols].head(8).round(3).to_string())

[미학습 카테고리 제외 후 TEST] n=1250  R2=-1.2774  RMSE=1.8423

[잔차 절댓값 상위 8건]
             category  log_views    pred  resid
5684            Music     19.233  13.556  5.677
5371            Music     18.817  13.410  5.407
5235            Music     18.624  13.483  5.141
6078    Entertainment     17.522  12.421  5.101
5456    Entertainment     17.700  12.689  5.011
5379            Music     18.752  13.813  4.939
5710  News & Politics     15.932  11.097  4.835
6068            Music     18.362  13.545  4.817


In [6]:
# 상위 1% 극단 잔차 제거 시 R2 변화 (RMSE 지배 여부)
thr = test_eval["resid"].abs().quantile(0.99)
trimmed = test_eval[test_eval["resid"].abs() <= thr]
r2_t = r2_score(trimmed["log_views"], trimmed["pred"])
print(f"극단 1% 제거 후 TEST R2 = {r2_t:.4f}  (제거 전 {r2:.4f})")

극단 1% 제거 후 TEST R2 = -1.3387  (제거 전 -1.2774)


## α. 랜덤 split 비교 — 시간 split 자체의 영향 분리
같은 모델을 **랜덤 80/20**으로 평가. 랜덤에선 정상인데 시간 split만 음수면
→ '시간에 따른 분포 변화'가 원인이라는 직접 증거.

In [7]:
from sklearn.model_selection import train_test_split
rtr, rte = train_test_split(features.dropna(subset=["publish_time"]),
                            test_size=0.2, random_state=42)
# 미학습 카테고리 제거 정합
rte = rte[rte["category"].isin(set(rtr["category"].unique()))]
rols = smf.ols(formula, data=rtr).fit()
rpred = rols.predict(rte)
print(f"[랜덤 split TEST] R2={r2_score(rte['log_views'], rpred):.4f}  "
      f"RMSE={np.sqrt(mean_squared_error(rte['log_views'], rpred)):.4f}")
print(f"[시간 split TEST] R2={r2:.4f} (미학습 제외) / 원본 -1.28")

[랜덤 split TEST] R2=0.1191  RMSE=1.7517
[시간 split TEST] R2=-1.2774 (미학습 제외) / 원본 -1.28


---
### 진단 결론
- **A 분포 변화 (주범)**: train 평균 log_views 12.73 → test 14.30 (+1.56), 표준편차 1.80→1.22로 축소. 뒤 시기(2018-03~06) 트렌딩 영상이 체계적으로 고조회수 → 모델이 test를 일괄 과소예측.
- **B 미학습 카테고리**: only_test=∅, only_train=∅. 레벨 미스매치 없음 → 무관.
- **C 극단 잔차**: 상위 1% 제거 시 R² −1.28→−1.35로 오히려 악화. 잔차 상위 8건 전부 양수(+4.8~5.8) → 소수 이상치가 아닌 전반적·체계적 under-prediction.
- **α 랜덤 vs 시간 (결정적)**: 랜덤 split R²=+0.116(정상) vs 시간 split R²=−1.28. split 방식만으로 갈림 → 음수 R²는 모델 결함이 아니라 **temporal drift(타깃 분포 이동)**.
- **종합**: 버그가 아닌 발견(finding). 성능 지표는 랜덤 split(+0.12) 기준으로 병기하고, 시간 split 음수는 4단계(ET 변환·드리프트) 스토리 근거로 보고.
- **다음 조치**: 우선순위 2 — VIF 처리(word_count 제거 권장) 후 재적합.